In [1]:
import os
import time
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

PROXY_URL = "http://localhost:4000"
MASTER_KEY = "sk-litellm-local-1234"


def make_model(tier: str, **overrides):
    """One factory, any tier alias from config.yaml."""
    return init_chat_model(
        tier,
        model_provider="openai",     # proxy is OpenAI-compatible regardless of upstream
        base_url=PROXY_URL,
        api_key=MASTER_KEY,
        max_tokens=overrides.pop("max_tokens", 500),
        **overrides,
    )



In [2]:

# ── 1. Basic call + routing/fallback verification ──
llm = make_model("fast")
resp = llm.invoke("What is 2+2?")
print("Answer:", resp.content)
print("Model that actually answered:", resp.response_metadata.get("model_name"))
# Confirms whether load balancing picked glm-5.2:free or deepseek-flash,
# or whether a fallback silently rerouted you — don't assume, check this every time.

Answer: 4
Model that actually answered: fast


In [3]:
# ── 2. Streaming ──
print("\nStreaming:")
for chunk in llm.stream("Count to 5 slowly."):
    print(chunk.content, end="", flush=True)
print()



Streaming:
Okay. I will count to five slowly.

One...

Two...

Three...

Four...

Five.


In [4]:
# ── 3. Caching proof — second identical call should be near-instant ──
prompt = "What is the capital of France?"

t0 = time.time()
r1 = llm.invoke(prompt)
t1 = time.time()
print(f"\nFirst call: {t1 - t0:.2f}s — {r1.content}")

t0 = time.time()
r2 = llm.invoke(prompt)
t1 = time.time()
print(f"Second call (cached): {t1 - t0:.2f}s — {r2.content}")
# If the second call isn't dramatically faster, caching isn't actually
# hitting — verify cache: true took effect after your last proxy restart.


First call: 4.72s — Paris.
Second call (cached): 0.02s — Paris.


In [5]:

# ── 4. Cost tracking per call ──
usage = resp.response_metadata.get("token_usage", {})
print("\nToken usage:", usage)
# litellm attaches per-request cost via response headers when the model
# is in its built-in cost map. Unmapped models (your OpenRouter stealth
# and custom slugs) return 0 — expected, not a bug, per the WARNING lines
# you saw at proxy startup earlier in this session.


Token usage: {'completion_tokens': 15, 'prompt_tokens': 90, 'total_tokens': 105, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 12, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0, 'cache_write_tokens': 0, 'cache_creation_tokens': 0}, 'cost': 8.064e-06, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 8.064e-06, 'upstream_inference_prompt_cost': 6.048e-06, 'upstream_inference_completions_cost': 2.016e-06}}


In [6]:
# ── 5. Explicit fallback trigger — force a failure, watch it reroute ──
llm_max = make_model("max")   # glm-5.3, mandates reasoning server-side
try:
    resp_max = llm_max.invoke(
        "Explain quantum entanglement in one sentence.",
        extra_body={"reasoning": {"enabled": False}},   # will be rejected by glm-5.3
    )
    print("\nFallback result model:", resp_max.response_metadata.get("model_name"))
    print(resp_max.content)
except Exception as e:
    print("\nFallback chain exhausted:", e)
    # If this fires, every model in max's fallback list (heavy, claude)
    # also rejected the request — check config.yaml's fallback chain,
    # don't assume the proxy silently handled it.


Fallback chain exhausted: Error code: 400 - {'error': {'message': 'litellm.BadRequestError: OpenrouterException - {"error":{"message":"Reasoning is mandatory for this endpoint and cannot be disabled.","code":400,"metadata":{"provider_name":null}}}. Received Model Group=max\nAvailable Model Group Fallbacks=[\'heavy\', \'claude\']\nError doing the fallback: litellm.BadRequestError: OpenrouterException - {"error":{"message":"Reasoning is mandatory for this endpoint and cannot be disabled.","code":400,"metadata":{"provider_name":null}}}. Received Model Group=heavy\nAvailable Model Group Fallbacks=[\'claude\', \'gemini\']\nError doing the fallback: litellm.NotFoundError: GeminiException - {\n  "error": {\n    "code": 404,\n    "message": "This model models/gemini-2.5-pro is no longer available to new users. Please update your code to use models/gemini-3.1-pro-preview for the latest features and improvements.",\n    "status": "NOT_FOUND"\n  }\n}\nNo fallback model group found for original m

In [8]:
# # ── 6. Cross-provider call — Anthropic through the same proxy ──
# llm_claude = make_model("claude")
# resp_claude = llm_claude.invoke("Say hi in 3 words.")
# print("\nClaude via proxy:", resp_claude.content)
# print("Confirmed model:", resp_claude.response_metadata.get("model_name"))


In [9]:
# ── 7. Rate-limit behavior — fire past a deployment's rpm cap on purpose ──
print("\nRate limit test (fast tier, rpm=20 per deployment):")
for i in range(5):
    r = llm.invoke(f"Say the number {i}.")
    print(f"  [{i}] {r.response_metadata.get('model_name')}: {r.content.strip()}")
    # Watch which underlying deployment answers each call — simple-shuffle
    # load balancing should distribute across both "fast" deployments.


Rate limit test (fast tier, rpm=20 per deployment):
  [0] fast: 0
  [1] fast: 1
  [2] fast: 2
  [3] fast: 3.3.3
  [4] fast: 4
